Meet-the-Neighbors Colab notebook.

This file is written in the same style as a Colab-exported Python notebook.
Open it in Google Colab, then use Runtime -> Run all after adjusting the
form fields.

# Meet-the-Neighbors: genome neighborhood VF prediction

This notebook runs the genome-only `meetneighbors predictvf` workflow on a
small upload set. It is intended for Google Colab, where RAM, disk, and GPU
time are limited.

## Inputs

Upload one pair of files per genome:

* One `.gff` annotation file.
* One matching protein FASTA file ending in `.faa` or `.fasta`.

The files are paired by shared basename. These are valid examples:

* `GenomeA.gff` and `GenomeA.faa`
* `GenomeB.gff` and `GenomeB.fasta`

By default, this notebook accepts up to 3 genomes. Change `MAX_GENOMES` in the
setup cell if you have a larger Colab runtime and want to allow more.

## Output

Only `neighborhood_based_predictions.tsv` is kept and downloaded. Uploaded
files, the cloned repository, and intermediate pipeline files are removed at
the end to save Colab disk space.

In [ ]:
#@title 1. Configure the run
#@markdown Edit these values before running the notebook.
#@markdown
#@markdown `REPO_URL` defaults to the public Meet-the-Neighbors repository.
#@markdown Use `REPO_REF` only if you want a specific branch, tag, or commit.
#@markdown Leave it blank to use the repository default branch.
REPO_URL = "https://github.com/mcn3159/meet-the-neighbors.git" #@param {type:"string"}
REPO_REF = "" #@param {type:"string"}
#@markdown
#@markdown Colab resources are limited. Keep `MAX_GENOMES` small unless you
#@markdown know the runtime has enough RAM and disk for a larger run.
MAX_GENOMES = 3 #@param {type:"integer"}
#@markdown
#@markdown Conservative defaults are used for typical free Colab runtimes.
threads = 2 #@param {type:"integer"}
mem_gb = 12 #@param {type:"integer"}
glm_batch_size = 50 #@param {type:"integer"}

from pathlib import Path
import os
import shutil
import subprocess
import sys

WORK_ROOT = Path("/content/meetneighbors_colab")
REPO_DIR = WORK_ROOT / "meet-the-neighbors"
UPLOAD_DIR = WORK_ROOT / "uploaded_genomes"
RUN_DIR = WORK_ROOT / "run"
OUTPUT_DIR = RUN_DIR / "mtn_output"
FINAL_DIR = WORK_ROOT / "final_results"
GENOME_TSV = RUN_DIR / "genome_pairs.tsv"
FINAL_OUTPUT = FINAL_DIR / "neighborhood_based_predictions.tsv"

print("Run configuration")
print(f"  repo: {REPO_URL}")
print(f"  ref: {REPO_REF or '(default branch)'}")
print(f"  max genomes: {MAX_GENOMES}")
print(f"  threads: {threads}")
print(f"  memory: {mem_gb} GB")
print(f"  gLM batch size: {glm_batch_size}")

## Install dependencies

This cell clones Meet-the-Neighbors, installs the Python package, and installs
the command-line tools required by the pipeline.

The install can take several minutes. If the runtime disconnects or restarts,
rerun the notebook from this cell.

In [ ]:
#@title 2. Install Meet-the-Neighbors, MMseqs2, and Foldseek
#@markdown This step installs the package and external tools in the Colab
#@markdown runtime. It is safe to rerun after a runtime reset.

def run_cmd(cmd, cwd=None):
    """Run a command and echo it in a notebook-friendly way."""
    printable = " ".join(str(part) for part in cmd)
    if cwd:
        print(f"$ cd {cwd} && {printable}")
    else:
        print(f"$ {printable}")
    subprocess.run([str(part) for part in cmd], cwd=cwd, check=True)


def ensure_miniforge():
    """Install Miniforge if conda is not already available."""
    if shutil.which("conda"):
        print("conda is already available")
        return

    installer = Path("/content/Miniforge3-Linux-x86_64.sh")
    url = (
        "https://github.com/conda-forge/miniforge/releases/latest/download/"
        "Miniforge3-Linux-x86_64.sh"
    )
    print("Installing Miniforge so mmseqs2 and foldseek can be installed.")
    run_cmd(["wget", "-qnc", url, "-O", installer])
    run_cmd(["bash", installer, "-bfp", "/usr/local"])
    os.environ["PATH"] = "/usr/local/bin:" + os.environ.get("PATH", "")
    if shutil.which("mamba"):
        run_cmd(["mamba", "config", "--set", "auto_update_conda", "false"])
    elif shutil.which("conda"):
        run_cmd(["conda", "config", "--set", "auto_update_conda", "false"])


def install_bio_tools():
    """Install mmseqs2 and foldseek through conda or mamba."""
    ensure_miniforge()
    installer = "mamba" if shutil.which("mamba") else "conda"
    run_cmd([
        installer,
        "install",
        "-y",
        "-c",
        "conda-forge",
        "-c",
        "bioconda",
        "mmseqs2",
        "foldseek",
    ])


WORK_ROOT.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

run_cmd(["git", "clone", REPO_URL, REPO_DIR])
if REPO_REF.strip():
    run_cmd(["git", "fetch", "--all", "--tags"], cwd=REPO_DIR)
    run_cmd(["git", "checkout", REPO_REF.strip()], cwd=REPO_DIR)

run_cmd([sys.executable, "-m", "pip", "install", "."], cwd=REPO_DIR)
install_bio_tools()

missing_tools = [
    tool for tool in ("meetneighbors", "mmseqs", "foldseek")
    if shutil.which(tool) is None
]
if missing_tools:
    raise RuntimeError(
        "The install finished, but these tools were not found on PATH: "
        + ", ".join(missing_tools)
    )

print("Install complete.")
print(f"meetneighbors: {shutil.which('meetneighbors')}")
print(f"mmseqs: {shutil.which('mmseqs')}")
print(f"foldseek: {shutil.which('foldseek')}")

## Upload genome files

Drag and drop all files together when prompted.

Each genome must have exactly one `.gff` and exactly one matching protein FASTA
file. Protein FASTA files may end in `.faa` or `.fasta`.

The notebook pairs files by basename, so `GenomeA.gff` pairs with
`GenomeA.faa`, and `GenomeB.gff` pairs with `GenomeB.fasta`.

In [ ]:
#@title 3. Upload `.gff` and `.faa`/`.fasta` genome pairs
#@markdown Click the upload button and select all genome files for this run.
#@markdown Upload no more than `MAX_GENOMES` `.gff` files and their matching
#@markdown protein FASTA files.

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError(
        "This upload cell is intended for Google Colab, where "
        "`google.colab.files.upload()` is available."
    ) from exc

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise ValueError("No files were uploaded.")

uploaded_paths = []
for uploaded_name in uploaded:
    source = Path(uploaded_name)
    normalized_name = source.stem + source.suffix.lower()
    destination = UPLOAD_DIR / normalized_name
    if destination.exists():
        raise ValueError(
            f"Duplicate uploaded filename after extension normalization: "
            f"{destination.name}"
        )
    shutil.move(str(source), destination)
    uploaded_paths.append(destination)

print("Uploaded files:")
for path in uploaded_paths:
    print(f"  {path.name}")

## Validate uploaded genome pairs

This cell checks the files before the long pipeline starts.

It fails early if:

* A file has an unsupported extension.
* A `.gff` does not have a matching `.faa` or `.fasta`.
* A genome has both `.faa` and `.fasta` uploaded.
* More than `MAX_GENOMES` genomes were uploaded.

In [ ]:
#@title 4. Validate genome pairs and build `genome_pairs.tsv`
#@markdown This creates the three-column TSV used by `meetneighbors --genome_tsv`.
#@markdown The columns are genome name, protein FASTA path, and GFF path.

PROTEIN_EXTENSIONS = {".faa", ".fasta"}
SUPPORTED_EXTENSIONS = PROTEIN_EXTENSIONS | {".gff"}


def uploaded_file_stem(path):
    suffix = path.suffix.lower()
    return path.name[: -len(suffix)]


def detect_genome_pairs(paths, max_genomes):
    gff_by_stem = {}
    proteins_by_stem = {}
    unsupported = []

    for path in paths:
        suffix = path.suffix.lower()
        stem = uploaded_file_stem(path)
        if suffix not in SUPPORTED_EXTENSIONS:
            unsupported.append(path.name)
        elif suffix == ".gff":
            if stem in gff_by_stem:
                raise ValueError(f"Duplicate GFF files for genome '{stem}'.")
            gff_by_stem[stem] = path
        else:
            proteins_by_stem.setdefault(stem, []).append(path)

    if unsupported:
        raise ValueError(
            "Unsupported uploaded file extension(s): "
            + ", ".join(sorted(unsupported))
            + ". Upload only .gff, .faa, or .fasta files."
        )

    stems = sorted(set(gff_by_stem) | set(proteins_by_stem))
    if not stems:
        raise ValueError("No genome files were detected.")
    if len(gff_by_stem) > max_genomes:
        raise ValueError(
            f"Uploaded {len(gff_by_stem)} genomes, but MAX_GENOMES is "
            f"{max_genomes}. Increase MAX_GENOMES only if your Colab runtime "
            "has enough resources."
        )

    pairs = []
    errors = []
    for stem in stems:
        gff = gff_by_stem.get(stem)
        proteins = proteins_by_stem.get(stem, [])
        if gff is None:
            errors.append(f"Missing .gff for '{stem}'.")
        if len(proteins) == 0:
            errors.append(f"Missing .faa or .fasta protein FASTA for '{stem}'.")
        elif len(proteins) > 1:
            protein_names = ", ".join(path.name for path in proteins)
            errors.append(
                f"Multiple protein FASTA files for '{stem}': {protein_names}. "
                "Upload exactly one of .faa or .fasta."
            )
        if gff is not None and len(proteins) == 1:
            pairs.append((stem, proteins[0], gff))

    if errors:
        raise ValueError("\n".join(errors))
    if len(pairs) > max_genomes:
        raise ValueError(
            f"Detected {len(pairs)} genome pairs, but MAX_GENOMES is "
            f"{max_genomes}."
        )
    return pairs


genome_pairs = detect_genome_pairs(uploaded_paths, MAX_GENOMES)
RUN_DIR.mkdir(parents=True, exist_ok=True)
with GENOME_TSV.open("w") as handle:
    for genome_name, protein_path, gff_path in genome_pairs:
        handle.write(f"{genome_name}\t{protein_path}\t{gff_path}\n")

print(f"Detected {len(genome_pairs)} genome pair(s):")
for genome_name, protein_path, gff_path in genome_pairs:
    print(f"  {genome_name}: {gff_path.name} + {protein_path.name}")
print(f"Wrote {GENOME_TSV}")

## Run Meet-the-Neighbors

This step can take a while because it computes protein language model and
genomic language model embeddings. GPU runtimes are recommended when available.

The command uses `predictvf` in genome-only mode through `--genome_tsv`.

In [ ]:
#@title 5. Run `meetneighbors predictvf`
#@markdown The notebook uses `--memory_optimize` and `--remove_temp` to reduce
#@markdown Colab disk and RAM pressure.

try:
    import torch
    gpu = 1 if torch.cuda.is_available() else 0
except Exception:
    gpu = 0

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predict_command = [
    "meetneighbors",
    "predictvf",
    "--genome_tsv",
    GENOME_TSV,
    "--out",
    OUTPUT_DIR,
    "--threads",
    threads,
    "--mem",
    mem_gb,
    "--glm_bs",
    glm_batch_size,
    "--gpu",
    gpu,
    "--memory_optimize",
    "--remove_temp",
]

print("Running Meet-the-Neighbors with this command:")
print(" ".join(str(part) for part in predict_command))
run_cmd(predict_command)

## Download the final prediction table

This notebook intentionally keeps only one output file:

`neighborhood_based_predictions.tsv`

All other pipeline files are treated as intermediates.

In [ ]:
#@title 6. Save, preview, and download `neighborhood_based_predictions.tsv`
#@markdown The final TSV is copied into a small results directory before
#@markdown cleanup starts.

import pandas as pd

pipeline_output = OUTPUT_DIR / "neighborhood_based_predictions.tsv"
if not pipeline_output.exists():
    raise FileNotFoundError(
        f"Expected output was not found: {pipeline_output}. "
        "Check the previous cell logs for the pipeline error."
    )

FINAL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(pipeline_output, FINAL_OUTPUT)

print(f"Final output saved to {FINAL_OUTPUT}")
preview_df = pd.read_csv(FINAL_OUTPUT, sep="\t")
print(f"Rows: {preview_df.shape[0]}, columns: {preview_df.shape[1]}")
display(preview_df.head())

files.download(str(FINAL_OUTPUT))

## Cleanup

This cell removes uploaded genome files, the cloned repository, and pipeline
intermediate files. The final TSV remains available at:

`/content/meetneighbors_colab/final_results/neighborhood_based_predictions.tsv`

Rerun the install and upload cells if you want to start a new analysis.

In [ ]:
#@title 7. Clean up intermediate files
#@markdown After this cell, only the copied final TSV is retained under
#@markdown `/content/meetneighbors_colab/final_results`.

for path in (UPLOAD_DIR, RUN_DIR, REPO_DIR):
    if path.exists():
        shutil.rmtree(path)

print("Cleanup complete.")
print(f"Kept final output: {FINAL_OUTPUT}")